# 08 - Care Management Targeting Simulation

if care management outreach only has capacity for the top-N highest-risk members, how much of total future cost does that actually capture -- and does the ML model beat something much simpler? Three targeting strategies are compared: random selection, ranking by **prior-year spend alone** (a naive but common real-world heuristic -- no model needed), and ranking by the trained model's **risk score**. The prior-spend comparison is the one that actually answers whether the model earns its complexity, not just whether it beats doing nothing.

In [ ]:
import json

import joblib
import pandas as pd

LABEL = "high_cost"
MODEL_NAME = "xgboost"
CAPACITY_LEVELS = [0.01, 0.05, 0.10]  # top 1% / 5% / 10% of the cohort

FEATURE_COLS = [
    "inpatient_admits", "outpatient_visits", "carrier_claims",
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "has_alzheimers", "has_chf", "has_ckd", "has_cancer", "has_copd",
    "has_depression", "has_diabetes", "has_ihd", "has_osteoporosis",
    "has_ra_oa", "has_stroke_tia", "has_esrd",
    "age", "is_male", "race_black", "race_other", "race_hispanic",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]


In [ ]:
# score every member in the cohort with the trained risk model
# (the model only ever sees 2008-2009 features -- it has no way to see
# total_paid_2010, so scoring on it isn't leakage)
model = joblib.load(f"../data/processed/models/{MODEL_NAME}_{LABEL}.joblib")
df = pd.read_parquet("../data/processed/features_labeled.parquet")

df["risk_score"] = model.predict_proba(df[FEATURE_COLS])[:, 1]

# "prior-year spend" heuristic: rank by 2008-2009 feature-window spend alone,
# with no model at all -- the simplest thing a payer could already be doing
df["prior_year_spend"] = df["inpatient_paid"] + df["outpatient_paid"] + df["carrier_paid"]


In [ ]:
# use the SAME frozen threshold notebook 05 derived from training data only
# (not the population-level high_cost column) so "who's really high-cost"
# stays consistent with what the model was trained to predict
with open("../data/processed/models/high_cost_threshold.json") as f:
    frozen_threshold = json.load(f)["cost_threshold"]

df["high_cost_frozen"] = (df["total_paid_2010"] >= frozen_threshold).astype(int)
print(f"frozen high_cost threshold (from notebook 05's training split): ${frozen_threshold:,.0f}")
print(f"members above it: {df['high_cost_frozen'].sum():,} ({df['high_cost_frozen'].mean():.1%} of cohort)")


In [ ]:
# small helper: capture rate for one targeting strategy at one capacity level
def capture_stats(df, rank_col, capacity_n, seed=42):
    if rank_col == "random":
        targeted = df.sample(n=capacity_n, random_state=seed)
    else:
        targeted = df.sort_values(rank_col, ascending=False).head(capacity_n)
    return {
        "members_targeted": capacity_n,
        "spend_captured": targeted["total_paid_2010"].sum(),
        "spend_share": targeted["total_paid_2010"].sum() / df["total_paid_2010"].sum(),
        "high_cost_captured": targeted["high_cost_frozen"].sum(),
        "high_cost_share": targeted["high_cost_frozen"].sum() / df["high_cost_frozen"].sum(),
    }


In [ ]:
# compare all three strategies at each capacity level
strategies = {
    "random": "random",
    "prior_year_spend": "prior_year_spend",
    "ml_risk_score": "risk_score",
}

rows = []
for capacity_pct in CAPACITY_LEVELS:
    capacity_n = round(len(df) * capacity_pct)
    for strategy_name, rank_col in strategies.items():
        stats = capture_stats(df, rank_col, capacity_n)
        stats["capacity_pct"] = capacity_pct
        stats["strategy"] = strategy_name
        rows.append(stats)

comparison = pd.DataFrame(rows)[
    ["capacity_pct", "strategy", "members_targeted", "spend_captured", "spend_share", "high_cost_captured", "high_cost_share"]
]
comparison


**does the model actually help, beyond just ranking by last year's bill?** lift vs. random targeting is the easy win; lift vs. prior-year spend is the number that actually justifies building a model instead of just sorting by last year's cost.

In [ ]:
pivot = comparison.pivot(index="capacity_pct", columns="strategy", values="spend_share")
pivot["lift_vs_random"] = pivot["ml_risk_score"] / pivot["random"]
pivot["lift_vs_prior_spend"] = pivot["ml_risk_score"] / pivot["prior_year_spend"]
pivot


**the top-1% headline number**, in the dollars-and-members format a stakeholder actually reads.

In [ ]:
top1 = comparison[comparison["capacity_pct"] == 0.01].set_index("strategy")
top1_display = top1[["members_targeted", "spend_captured", "spend_share"]].copy()
top1_display["spend_captured"] = top1_display["spend_captured"].map(lambda x: f"${x:,.0f}")
top1_display["spend_share"] = top1_display["spend_share"].map(lambda x: f"{x:.1%}")
top1_display


In [ ]:
# save the strategy comparison + per-member risk scores for the dashboard
comparison.to_parquet("../data/processed/targeting_comparison.parquet", index=False)

member_scores = df[["desynpuf_id", "risk_score", "total_paid_2010", "high_cost_frozen"]].copy()
member_scores["risk_tier"] = pd.qcut(
    member_scores["risk_score"], q=[0, 0.5, 0.8, 0.95, 1.0],
    labels=["low", "medium", "high", "very_high"],
)
member_scores.to_parquet("../data/processed/member_risk_scores.parquet", index=False)

print("wrote targeting_comparison.parquet and member_risk_scores.parquet")
